- Test script to get the nyc Green taxi data 2024
- to test different functions and features for the main script.

In [ ]:
import requests
from joblib import load, dump
from tqdm import tqdm
import pandas as pd




In [22]:
# Get the data from nyc website.

file_name = "green_tripdata_2024-03.parquet"

def download_dataset(file_name):
    url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/{file_name}"
    response = requests.get(url, stream=True)
    #save_path = pathlib.Path(os.getcwd()).joinpath('data',file_name)
    save_path = f"./data/{file_name}"
    print(save_path)
    with open(save_path, 'wb') as i_file:
        for chunk in tqdm(response.iter_content(), 
                          desc=f"Downloading {file_name}",
                          postfix=f"saving the dataset to path:{save_path}",
                          total = int(response.headers["Content-Length"])):
            i_file.write(chunk)

download_dataset(file_name)

./data/green_tripdata_2024-03.parquet


In [25]:
march_df = pd.read_parquet("data/green_tripdata_2024-03.parquet")
march_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57457 entries, 0 to 57456
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   VendorID               57457 non-null  int32         
 1   lpep_pickup_datetime   57457 non-null  datetime64[us]
 2   lpep_dropoff_datetime  57457 non-null  datetime64[us]
 3   store_and_fwd_flag     55360 non-null  object        
 4   RatecodeID             55360 non-null  float64       
 5   PULocationID           57457 non-null  int32         
 6   DOLocationID           57457 non-null  int32         
 7   passenger_count        55360 non-null  float64       
 8   trip_distance          57457 non-null  float64       
 9   fare_amount            57457 non-null  float64       
 10  extra                  57457 non-null  float64       
 11  mta_tax                57457 non-null  float64       
 12  tip_amount             57457 non-null  float64       
 13  t

In [27]:
with open('model/model.pkl', 'rb') as f_out:
    model = load(f_out)

with open('model/dict_vectorizer.pkl', 'rb') as f_out:
    dv = load(f_out)

/opt/conda/envs/monitor/lib/python3.11/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LinearRegression from version 1.6.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/conda/envs/monitor/lib/python3.11/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator DictVectorizer from version 1.6.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:
'''
In this case I already have the model and dict_vectorizer saved in the model folder. so, we use the voculabary as train and val df. 
However, to generate monitoring metrics I need prediciton columns in the dataframe. so I will use the modle to
to predict and use this column in Evidently to generat the metrics.
- We save the reference dataset and validation dataset for monitoring and metrics generation.
'''

categorical = ['PULocationID', 'DOLocationID']
numerical = ['trip_distance']
train_data = march_df[:30000]
val_data = march_df[30000:]


def prep_data_and_predict(df, dv, model):
    print(f"preparing dataframe..")
    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60
    df = df[(df.duration >= 0) & (df.duration <= 60)].copy()

    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    dicts = df[categorical + numerical].to_dict(orient='records')

    print("transforming the data..")
    X = dv.transform(dicts)
    print("prediction started..")
    temp_pred = model.predict(X)
    df['prediction'] = temp_pred
    print("predictons are done..")
    return df

train_pred_df = prep_data_and_predict(train_data, dv, model)
val_pred_df = prep_data_and_predict(val_data, dv, model)

preparing dataframe..
transforming the data..


/tmp/ipykernel_18561/4064457436.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
/tmp/ipykernel_18561/4064457436.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['duration'] = df.duration.dt.total_seconds() / 60


prediction started..
predictons are done..
preparing dataframe..
transforming the data..


/tmp/ipykernel_18561/4064457436.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
/tmp/ipykernel_18561/4064457436.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['duration'] = df.duration.dt.total_seconds() / 60


prediction started..
predictons are done..


In [33]:
train_pred_df.to_parquet('data/reference_train.parquet')
val_pred_df.to_parquet('data/current_val.parquet')